In [1]:
import pandas as pd

# Part 1: Returns

In [2]:
df = pd.DataFrame({
    "Date": pd.date_range("2026-01-01", periods=6, freq="D"),
    "A": [100, 102, 101, 105, 103, 108],
    "B": [200, 198, 202, 204, 201, 207]
})

df = df.set_index("Date")

df

,A,B
Date,,
2026-01-01,100,200
2026-01-02,102,198
2026-01-03,101,202
2026-01-04,105,204
2026-01-05,103,201
2026-01-06,108,207


In [3]:
returns = df.pct_change().dropna()
returns

,A,B
Date,,
2026-01-02,0.020000,-0.010000
2026-01-03,-0.009804,0.020202
2026-01-04,0.039604,0.009901
2026-01-05,-0.019048,-0.014706
2026-01-06,0.048544,0.029851


In [4]:
# To calculate returns over a 3-row interval:
returns_3d = df.pct_change(periods=3)
returns_3d

,A,B
Date,,
2026-01-01,NaN,NaN
2026-01-02,NaN,NaN
2026-01-03,NaN,NaN
2026-01-04,0.050000,0.020000
2026-01-05,0.009804,0.015152
2026-01-06,0.069307,0.024752


In [5]:
daily_returns = (df / df.shift(1)) - 1
daily_returns = daily_returns.dropna()
daily_returns

,A,B
Date,,
2026-01-02,0.020000,-0.010000
2026-01-03,-0.009804,0.020202
2026-01-04,0.039604,0.009901
2026-01-05,-0.019048,-0.014706
2026-01-06,0.048544,0.029851


In [6]:
# 3. Create a new DataFrame called df2 containing the original prices and two new columns: A_return and B_return.
df2 = df.copy()
df2["A_return"] = (df2["A"] / df2["A"].shift(1)) - 1
df2["B_return"] = (df2["B"] / df2["B"].shift(1)) - 1

df2

,A,B,A_return,B_return
Date,,,,
2026-01-01,100,200,NaN,NaN
2026-01-02,102,198,0.020000,-0.010000
2026-01-03,101,202,-0.009804,0.020202
2026-01-04,105,204,0.039604,0.009901
2026-01-05,103,201,-0.019048,-0.014706
2026-01-06,108,207,0.048544,0.029851


In [7]:
# cleaner way:
df2 = df.copy()
df2[["A_return", "B_return"]] = df.pct_change()

In [8]:
date_rq = df2[df2["A_return"] == df2["A_return"].min()].index
date_rq

DatetimeIndex(['2026-01-05'], dtype='datetime64[ns]', name='Date', freq=None)

In [9]:
# A simpler way to retrieve the date is:
df2["A_return"].idxmin()

Timestamp('2026-01-05 00:00:00')

**idxmin() or idxmax() to fetch the index**

# Part 2: shift()

### **shift()**

In [10]:
import pandas as pd

df = pd.DataFrame({
    "Price": [100, 102, 101, 105, 103, 108]
}, index=pd.date_range("2026-01-01", periods=6, freq="D"))

df

,Price
2026-01-01,100
2026-01-02,102
2026-01-03,101
2026-01-04,105
2026-01-05,103
2026-01-06,108


In [11]:
# Step 1: Calculate daily returns
df["Return"] = df["Price"].pct_change()

# Step 2: Create a one-day lag of returns
df["Return_Lag1"] = df["Return"].shift(1)

df["t_plus_2d"] = df["Return"].shift(-2)

# Step 3: Compare today's return with yesterday's
df["Return_Difference"] = df["Return"] - df["Return_Lag1"]

df

,Price,Return,Return_Lag1,t_plus_2d,Return_Difference
2026-01-01,100,NaN,NaN,-0.009804,NaN
2026-01-02,102,0.020000,NaN,0.039604,NaN
2026-01-03,101,-0.009804,0.020000,-0.019048,-0.029804
2026-01-04,105,0.039604,-0.009804,0.048544,0.049408
2026-01-05,103,-0.019048,0.039604,NaN,-0.058652
2026-01-06,108,0.048544,-0.019048,NaN,0.067591


In [12]:
## Practise

df = pd.DataFrame({
    "Price": [100, 102, 101, 105, 103, 108],
    "Volume": [1000, 1200, 900, 1500, 1100, 1600]
}, index=pd.date_range("2026-01-01", periods=6, freq="D"))

df

,Price,Volume
2026-01-01,100,1000
2026-01-02,102,1200
2026-01-03,101,900
2026-01-04,105,1500
2026-01-05,103,1100
2026-01-06,108,1600


In [13]:
# Q1. Create a Return column containing daily percentage returns from Price.
df["Return"] = df["Price"].pct_change()

# Q2. Create a Previous_Price column containing the previous day's price.
df["Previous_Price"] = df["Price"].shift(1)

# Q3. Create a Volume_Change column showing today's volume minus yesterday's volume.
df["Volume_Change"] = df["Volume"] - df["Volume"].shift(1)

df

,Price,Volume,Return,Previous_Price,Volume_Change
2026-01-01,100,1000,NaN,NaN,NaN
2026-01-02,102,1200,0.020000,100.0,200.0
2026-01-03,101,900,-0.009804,102.0,-300.0
2026-01-04,105,1500,0.039604,101.0,600.0
2026-01-05,103,1100,-0.019048,105.0,-400.0
2026-01-06,108,1600,0.048544,103.0,500.0


# Part 3: Rolling and Expanding Windows

In [14]:
import pandas as pd


df = pd.DataFrame({
    "Price": [100, 102, 101, 105, 103, 108],
    "Volume": [1000, 1200, 900, 1500, 1100, 1600]
}, index=pd.date_range("2026-01-01", periods=6, freq="D"))

df

,Price,Volume
2026-01-01,100,1000
2026-01-02,102,1200
2026-01-03,101,900
2026-01-04,105,1500
2026-01-05,103,1100
2026-01-06,108,1600


### Rolling mean  

In [15]:
#Calculate the average price over the last 3 observations.
df["Rolling_Mean"] = df["Price"].rolling(window=3).mean()

df

,Price,Volume,Rolling_Mean
2026-01-01,100,1000,NaN
2026-01-02,102,1200,NaN
2026-01-03,101,900,101.000000
2026-01-04,105,1500,102.666667
2026-01-05,103,1100,103.000000
2026-01-06,108,1600,105.333333


### Rolling volatility

In [16]:
df["Return"] = df["Price"].pct_change()

df["Rolling Volatility"] = df["Return"].rolling(3).std()

# annualize daily volatility
import numpy as np
df["Annualized_Volatility"] = df["Rolling Volatility"] * np.sqrt(250)

df

,Price,Volume,Rolling_Mean,Return,Rolling Volatility,Annualized_Volatility
2026-01-01,100,1000,NaN,NaN,NaN,NaN
2026-01-02,102,1200,NaN,0.020000,NaN,NaN
2026-01-03,101,900,101.000000,-0.009804,NaN,NaN
2026-01-04,105,1500,102.666667,0.039604,0.024879,0.393368
2026-01-05,103,1100,103.000000,-0.019048,0.031535,0.498606
2026-01-06,108,1600,105.333333,0.048544,0.036716,0.580535


### Expanding mean

In [17]:
df["Expanding_Mean"] = (df["Price"].expanding().mean())

df

,Price,Volume,Rolling_Mean,Return,Rolling Volatility,Annualized_Volatility,Expanding_Mean
2026-01-01,100,1000,NaN,NaN,NaN,NaN,100.000000
2026-01-02,102,1200,NaN,0.020000,NaN,NaN,101.000000
2026-01-03,101,900,101.000000,-0.009804,NaN,NaN,101.000000
2026-01-04,105,1500,102.666667,0.039604,0.024879,0.393368,102.000000
2026-01-05,103,1100,103.000000,-0.019048,0.031535,0.498606,102.200000
2026-01-06,108,1600,105.333333,0.048544,0.036716,0.580535,103.166667


In [18]:
df["Expanding_Volume"] = df["Volume"].expanding().mean()

df

,Price,Volume,Rolling_Mean,Return,Rolling Volatility,Annualized_Volatility,Expanding_Mean,Expanding_Volume
2026-01-01,100,1000,NaN,NaN,NaN,NaN,100.000000,1000.000000
2026-01-02,102,1200,NaN,0.020000,NaN,NaN,101.000000,1100.000000
2026-01-03,101,900,101.000000,-0.009804,NaN,NaN,101.000000,1033.333333
2026-01-04,105,1500,102.666667,0.039604,0.024879,0.393368,102.000000,1150.000000
2026-01-05,103,1100,103.000000,-0.019048,0.031535,0.498606,102.200000,1140.000000
2026-01-06,108,1600,105.333333,0.048544,0.036716,0.580535,103.166667,1216.666667


# Part 4: Resampling

In [19]:
df = pd.DataFrame({
    "Price": [100, 102, 101, 105, 103, 108],
    "Volume": [1000, 1200, 900, 1500, 1100, 1600]
}, index=pd.date_range("2026-01-01", periods=6, freq="D"))

df

,Price,Volume
2026-01-01,100,1000
2026-01-02,102,1200
2026-01-03,101,900
2026-01-04,105,1500
2026-01-05,103,1100
2026-01-06,108,1600


In [20]:
# Suppose you want the last observed price and total volume for each week.

weekly = df.resample("W").agg({
    "Price": "last",
    "Volume": "sum"
})

weekly

,Price,Volume
2026-01-04,105,4600
2026-01-11,108,2700


Notice that the first group contains January 1–4, and the second contains January 5–6. The final label is January 11 because the weekly period ends on Sunday.

In [21]:
monthly_price = df["Price"].resample("BM").last()
monthly_price

2026-01-30    108
Freq: BM, Name: Price, dtype: int64

### Practise

In [22]:
import yfinance as yf

tickers = ["IXN", "IXC"]

data = yf.download(
    tickers,
    start="2015-01-01",
    end="2026-10-21",
    auto_adjust=True,   # Close is adjusted for splits/dividends
    progress=False,
)["Close"]

data = data.dropna()   # keep only dates where both have data


data.tail()

Ticker,IXC,IXN
Date,,
2026-10-14,62.090000,142.649994
2026-10-15,61.380001,141.449997
2026-10-16,62.980000,137.369995
2026-10-19,62.410000,139.259995
2026-10-20,62.650002,138.770004


In [23]:
data.index.dtype

dtype('<M8[ns]')

In [24]:
monthly = data.resample("BM").last()
monthly.tail()

Ticker,IXC,IXN
Date,,
2026-06-30,49.130001,144.479996
2026-07-31,55.730000,132.679993
2026-08-31,58.180000,141.130005
2026-09-30,56.970001,148.220001
2026-10-30,62.650002,138.770004


In [25]:
monthly_returns = monthly.pct_change()
monthly_returns.tail()

Ticker,IXC,IXN
Date,,
2026-06-30,-0.065923,0.006001
2026-07-31,0.134337,-0.081672
2026-08-31,0.043962,0.063687
2026-09-30,-0.020798,0.050237
2026-10-30,0.099702,-0.063757


In [26]:
# Write code to calculate monthly returns from month-end prices, storing them in monthly_returns.

monthly_returns = data.resample("BM").last().pct_change()